# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR^2 mass spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library for Python.

### Dataset Source
This dataset is accessible via a Croissant schema URL describing its metadata and structure according to the [Croissant standard](https://mlcommons.org/croissant/).

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
We begin by loading the dataset metadata and record sets using `mlcroissant`. The example in this notebook is based on the FAIR^2 mass spectrometry dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the Croissant metadata object (not as a dict; use attributes)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List the available record sets and their fields. All references use their `@id` values, following the dataset schema.

In [ ]:
# Gather record sets by their @id
print('Record sets and their fields:')
record_sets = []

# The Croissant spec for mlcroissant gives access via metadata.record_sets
for rset in metadata.record_sets:
    print(f"- RecordSet Name: {rset.name}")
    print(f"  @id: {rset.id}")
    print("  Fields:")
    for field in rset.fields:
        print(f"    - {field.name} (@id: {field.id}, data_type: {field.data_type})")
    print()
    record_sets.append(rset.id)

if not record_sets:
    print('No RecordSets found. Check the Croissant schema or dataset documentation.')

## 3. Data Extraction
Load data from all found record sets into pandas DataFrames for further analysis. Use record set and field `@id`s from the overview above.

In [ ]:
# Extract each record set into a DataFrame, indexed by @id
dataframes = {}

for record_set_id in record_sets:
    try:
        print(f"Loading records for: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"  Loaded {len(df)} records, columns: {df.columns.tolist()}")
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"  Failed to load {record_set_id}: {e}")

# For demonstration, pick the first available record set if any
if record_sets:
    first_rs = record_sets[0]
    print(f"\nExample (columns for first RecordSet {first_rs}):")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Process a numeric field (column) for outlier removal, normalization, and grouping. Replace the field IDs as appropriate for your data by referencing the `@id`s found in the Data Overview.

In [ ]:
# Example: Choose a numeric field and a group field from the first record set's fields (update manually as appropriate)
# We recommend inspecting the columns (see above).

if not record_sets:
    print('No record sets loaded; cannot perform analysis.')
else:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]

    # Guess some field names: e.g., a coverage, abundance, peptide count, or MW field
    # You may need to update these field names based on the dataset output above
    candidate_numeric_fields = [
        col for col in df.columns 
        if any(substr in col.lower() for substr in ['abundance', 'coverage', 'peptide', 'mw', 'count', 'score', 'area', 'value', 'intensity']) and pd.api.types.is_numeric_dtype(df[col])
    ]
    group_fields = [
        col for col in df.columns if any(substr in col.lower() for substr in ['sample', 'condition', 'group', 'category'])
    ]

    numeric_field = None
    group_field = None
    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
        print(f"Selected numeric field: {numeric_field}")
    if group_fields:
        group_field = group_fields[0]
        print(f"Selected group field: {group_field}")

    if numeric_field is None:
        print("No numeric field found for analysis.")
    else:
        # Remove obvious outliers (e.g., negative values if not allowed)
        threshold = df[numeric_field].quantile(0.05) if df[numeric_field].min() < 0 else df[numeric_field].mean() + 3*df[numeric_field].std()
        filtered_df = df[df[numeric_field] < threshold]
        print(f"\nFiltered records with {numeric_field} below threshold ({threshold}): {len(filtered_df)} of {len(df)} records remain.")
        display(filtered_df[[numeric_field]].head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nExample normalization ({numeric_field}):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean for '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())

## 5. Visualization
Let's create one or more plots to illustrate the distribution of the selected numeric field(s) and their grouping.

For advanced visualizations, you can use libraries such as `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field is not None:
        # Boxplot by group field
        plt.figure(figsize=(10,4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- We successfully loaded and explored the FAIR^2 mass spectrometry dataset using the Croissant schema and the `mlcroissant` library.
- Record sets and field structures have been identified using their `@id` fields, and sample records were extracted for analysis.
- Simple exploratory analyses such as filtering, normalization, and group aggregation illustrated how you can work with real scientific datasets.
- You can extend this notebook with more advanced analyses, machine learning models, or domain-specific visualizations as needed!

For more details on record set structure, consult the dataset documentation or the schema at the Croissant URL above.